In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
    

In [2]:
X, y = make_classification(n_samples=100, n_features=4, n_classes=2, random_state=42)
df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(4)])
df['target'] = y

In [3]:
weights = np.zeros(X.shape[1])
intercept = 0.0

In [4]:
def predict_proba(X):
    y_pred = X @ weights + intercept
    return 1 / (1 + np.exp(-y_pred))

In [12]:
def predict(X, threshold=0.5):
    proba = predict_proba(X)
    return (proba >= threshold).astype(int)

In [6]:
def intercept_dv(y_pred, y_true):
    return np.mean(y_pred - y_true)

In [7]:
def weights_dv(y_pred, y_true, X):
    return (X.T @ (y_pred - y_true)) / len(y_true)

In [10]:
def train(X, y, lr = 0.01, max_itters=100):
    global weights, intercept
    for _ in range(max_itters):
        y_pred = predict_proba(X)
        w_dv = weights_dv(y_pred, y, X)
        b_dv = intercept_dv(y_pred, y)
        
        weights -= lr * w_dv
        intercept -= lr * b_dv

In [17]:
train(X, y)
y_pred = predict(X)
y_proba = predict_proba(X)


df_preds = pd.DataFrame({"probability": y_proba, "predicted_class": y_pred})
print(df_preds, "\nAccuracy:", accuracy_score(y, y_pred))

    probability  predicted_class
0      0.872996                1
1      0.252220                0
2      0.966169                1
3      0.011775                0
4      0.014225                0
..          ...              ...
95     0.035595                0
96     0.626101                1
97     0.266218                0
98     0.961344                1
99     0.973200                1

[100 rows x 2 columns] 
Accuracy: 0.97


In [18]:


model = LogisticRegression()
model.fit(X, y)

y_pred = model.predict(X)
y_proba = model.predict_proba(X)[:, 1]

df_preds = pd.DataFrame({"probability": y_proba, "predicted_class": y_pred})
print(df_preds, "\nAccuracy:", accuracy_score(y, y_pred))

    probability  predicted_class
0      0.986266                1
1      0.057436                0
2      0.998721                1
3      0.000233                0
4      0.000323                0
..          ...              ...
95     0.005728                0
96     0.682355                1
97     0.154687                0
98     0.995640                1
99     0.999105                1

[100 rows x 2 columns] 
Accuracy: 0.99


## **Okay so now that I got an Idea of how logistic regression works with logloss and sigmoid**
## The next step is making it into an SGD Classifier that uses
### - Stable logloss instead of normal logloss to prevent infinites and NANs
####  l = ln(1 + e^z) - yz or logaddexp(0, z) - y*z
### - Stable Sigmoid instead of normal sigmoid also to prevent explosions and NANs
####  if z >= 0  sigmoid(z) = 1 / (1 + e^-z)
####  if z < 0 sigmoid(z) = e^z/ (1 + e^z)
### - l2 regurlarization same as the one in SGD Regressor just usig logloss instead of mse
### - lr scheduler similar to the SGD Regressor
### - early stopping using logloss on validation
### - adding class_weights hyperparameter for imbalanced labels & decision_function() for debugging & ROC
### - Testing and final modifications before adding anything new